In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
os.chdir('/Users/jinghuang/Desktop/SSTP/Parse')

In [ ]:
def flatten_labels(label_tree, blanks=True):
    flat = []
    for category, tests in label_tree.items():
        if blanks: 
            flat.append((category,'', ''))
        else:
            flat.append((category,))
        if isinstance(tests, list):
            for t in tests:
                if blanks:
                    flat.append((category, t, ''))
                else:
                    flat.append((category, t))
        else:                       # dict
            for test_name, subs in tests.items():
                if subs:
                    for sub in subs:
                        flat.append((category, test_name, sub))
                else:
                    if blanks:
                        flat.append((category, test_name, ''))
                    else:
                        flat.append((category, test_name))
    return flat

def label_strings(flat_labels: list[tuple[str, str, str | None]]) -> list[str]:
    """Render each (cat, test, sub) triple as a single string for matching."""
    return [" | ".join(part for part in triple if part) for triple in flat_labels]

In [ ]:
# Get patient
patient = Path('case_outputs/tables_clean/00001.csv')
patient = pd.read_csv(patient)

# Get field_labels
fl_path = Path('instructions/field_labels.json')
fl_flat = flatten_labels(json.loads(fl_path.read_text()), blanks=False)

In [ ]:
idx1, idx2 = 0,0
res = []
while idx1 < patient.shape[0]-1:
    if idx2 < len(fl_flat):
        B = fl_flat[idx2]
    else:
        res.append((idx2, None))
        print(f"Not FOUND {idx1:02d} {idx2:02d}\t ")
        idx1 += 1
        idx2 = 0

    if len(B) == 1: # if category, needs to also match casewise 
        A = patient.iloc[idx1]['test_info']
        B0 = B[0].upper()
    else:
        A = A.lower()
        B0 = B[-1].lower()
    if B0 in A:
        res.append((idx2, idx1))
        print(f"WE FOUND {idx1:02d} {idx2:02d} {B}\t {patient.iloc[idx1].values}")
        idx1 += 1
        idx2 = 0
    else:
        idx2 += 1

In [ ]:
"""fl_flat = flatten_labels(json.loads(fl_path.read_text()))
multi_cols = pd.MultiIndex.from_tuples(
    fl_flat, names=["Category", "Test", "Subtest"]
)
data = np.empty((3, len(multi_cols)), dtype=object)
patient_no = 0
for test_name_idx, patient_row_idx in res:
    if patient_row_idx is not None:
        data[patient_no][test_name_idx] = patient.iloc[patient_row_idx]['Middle']
        data[patient_no+1][test_name_idx] = patient.iloc[patient_row_idx]['Left']
        data[patient_no+2][test_name_idx] = patient.iloc[patient_row_idx]['Right']

df = pd.DataFrame(data, columns=multi_cols)

file_name = 'report_with_multi_header.xlsx'

# Simply remove 'index=False'. This resolves the error.
with pd.ExcelWriter(file_name, engine="xlsxwriter") as writer:
    # dump the DataFrame
    df.to_excel(writer, sheet_name="Results")

    # grab the native XlsxWriter worksheet object
    ws = writer.sheets["Results"]

    ws.set_column(first_col=0,
                  last_col=len(df.columns) - 1,
                  width=20)


print(f"✅ Excel file '{file_name}' was created successfully (with index column).")"""

# ----------------------------------  SET-UP  ----------------------------------
fl_flat = flatten_labels(json.loads(fl_path.read_text()))
multi_cols = pd.MultiIndex.from_tuples(fl_flat,
                                       names=["Category", "Test", "Subtest"])

# ---- identify your patients ---------------------------------------------------
# Example: one CSV per patient in a list; adapt to your real source
patients = [patient]                      # <- replace with real list/iterator
patient_ids = [p.attrs.get("id", i) for i, p in enumerate(patients)]

# ---- allocate the backing array ----------------------------------------------
n_patients = len(patient_ids)
data = np.empty((n_patients * 3, len(multi_cols)), dtype=object)

# ---- fill it ------------------------------------------------------------------
for p_idx, p_df in enumerate(patients):
    base = p_idx * 3                      # first row (raw_score) for this patient
    for test_name_idx, patient_row_idx in res:
        if patient_row_idx is None:
            continue
        data[base    , test_name_idx] = p_df.iloc[patient_row_idx]["raw_score"]  # raw_score
        data[base + 1, test_name_idx] = p_df.iloc[patient_row_idx]["comments"]    # comments
        data[base + 2, test_name_idx] = p_df.iloc[patient_row_idx]["test_info"]   # test_info

# ---- build the row MultiIndex -------------------------------------------------
row_tuples = [
    (pid, label)
    for pid in patient_ids
    for label in ("raw_score", "comments", "test_info")
]
row_index = pd.MultiIndex.from_tuples(row_tuples,
                                      names=["Patient", "Field"])

df = pd.DataFrame(data, index=row_index, columns=multi_cols)

# ----------------------------------  EXPORT  ----------------------------------
file_name = "report_with_multi_header.xlsx"

with pd.ExcelWriter(file_name, engine="xlsxwriter") as writer:
    df.to_excel(writer, sheet_name="Results")          # keep the index; it's meaningful

    ws = writer.sheets["Results"]
    ws.set_column(first_col=0,
                  last_col=len(df.columns) - 1,
                  width=20)                            # uniform width (or auto-fit loop)

print(f"✅ Excel file '{file_name}' was created successfully.")


In [ ]:
fl_flat